In [1]:
import pandas as pd
import numpy as np

columns = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "B365H", "B365D", "B365A"]

seasons = []
for code in ["2223", "2324", "2425", "2526"]:
    s = pd.read_csv("https://www.football-data.co.uk/mmz4281/" + code + "/E0.csv")[columns]
    s["Season"] = code
    seasons.append(s)

data = pd.concat(seasons)
data["Date"] = pd.to_datetime(data["Date"], dayfirst=True)
data = data.sort_values("Date").reset_index(drop=True)

print(len(data), "matches")
print(data["Season"].value_counts())

1520 matches
Season
2223    380
2324    380
2425    380
2526    380
Name: count, dtype: int64


In [2]:
scored = {}
conceded = {}

h_scored = []
h_conceded = []
a_scored = []
a_conceded = []

for i in range(len(data)):
    home = data["HomeTeam"].iloc[i]
    away = data["AwayTeam"].iloc[i]
    hg = data["FTHG"].iloc[i]
    ag = data["FTAG"].iloc[i]

    # First time we see a team: give it empty lists
    for team in [home, away]:
        if team not in scored:
            scored[team] = []
            conceded[team] = []

    # 1. Features: form from the last 5 matches BEFORE this one
    if len(scored[home]) >= 5 and len(scored[away]) >= 5:
        h_scored.append(np.mean(scored[home][-5:]))
        h_conceded.append(np.mean(conceded[home][-5:]))
        a_scored.append(np.mean(scored[away][-5:]))
        a_conceded.append(np.mean(conceded[away][-5:]))
    else:
        h_scored.append(np.nan)
        h_conceded.append(np.nan)
        a_scored.append(np.nan)
        a_conceded.append(np.nan)

    # 2. AFTER the match: write down the result
    scored[home].append(hg)
    conceded[home].append(ag)
    scored[away].append(ag)
    conceded[away].append(hg)

data["h_scored"] = h_scored
data["h_conceded"] = h_conceded
data["a_scored"] = a_scored
data["a_conceded"] = a_conceded

print(data[["Date", "HomeTeam", "AwayTeam", "h_scored", "h_conceded", "a_scored", "a_conceded"]].iloc[[0, 50, 500, 1000]])

           Date        HomeTeam   AwayTeam  h_scored  h_conceded  a_scored  \
0    2022-08-05  Crystal Palace    Arsenal       NaN         NaN       NaN   
50   2022-09-03         Everton  Liverpool       0.8         1.2       3.0   
500  2023-11-25        Man City  Liverpool       3.0         1.4       2.2   
1000 2025-02-14        Brighton    Chelsea       1.2         2.0       1.8   

      a_conceded  
0            NaN  
50           1.2  
500          0.6  
1000         1.6  


In [3]:
data["missing"] = data["h_scored"].isna()

print(data.groupby("Season")["missing"].sum())

Season
2223    50
2324    15
2425     5
2526     5
Name: missing, dtype: int64


In [4]:
features = ["h_scored", "h_conceded", "a_scored", "a_conceded"]

train = data[data["Season"].isin(["2324", "2425"])].dropna(subset=features)
test = data[data["Season"] == "2526"].copy()

test[features] = test[features].fillna(train[features].mean())

print("Train:", len(train), "matches")
print("Test:", len(test), "matches")
print("Missing in test now:", test[features].isna().sum().sum())

Train: 740 matches
Test: 380 matches
Missing in test now: 0


In [5]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(train[features], train["FTR"])

print("Classes:", model.classes_)

probs = model.predict_proba(test[features])

test["ml_A"] = probs[:, 0]
test["ml_D"] = probs[:, 1]
test["ml_H"] = probs[:, 2]

print(test[["HomeTeam", "AwayTeam", "ml_H", "ml_D", "ml_A"]].head(10).round(3))

Classes: ['A' 'D' 'H']
           HomeTeam        AwayTeam   ml_H   ml_D   ml_A
1140      Liverpool     Bournemouth  0.450  0.271  0.279
1141    Aston Villa       Newcastle  0.344  0.267  0.389
1142       Brighton          Fulham  0.635  0.227  0.138
1143     Sunderland        West Ham  0.441  0.241  0.318
1144      Tottenham         Burnley  0.266  0.204  0.531
1145         Wolves        Man City  0.258  0.245  0.497
1146        Chelsea  Crystal Palace  0.423  0.228  0.349
1147  Nott'm Forest       Brentford  0.347  0.219  0.435
1148     Man United         Arsenal  0.373  0.238  0.389
1149          Leeds         Everton  0.194  0.217  0.589


In [6]:
# What actually happened
is_H = (test["FTR"] == "H").astype(int)
is_D = (test["FTR"] == "D").astype(int)
is_A = (test["FTR"] == "A").astype(int)

def grade(pH, pD, pA):
    brier = ((pH - is_H)**2 + (pD - is_D)**2 + (pA - is_A)**2).mean()
    p_actual = pH * is_H + pD * is_D + pA * is_A
    log_loss = -np.log(p_actual).mean()
    return round(brier, 4), round(log_loss, 4)

# Bookmaker (margin removed)
margin = 1 / test["B365H"] + 1 / test["B365D"] + 1 / test["B365A"]
book_H = (1 / test["B365H"]) / margin
book_D = (1 / test["B365D"]) / margin
book_A = (1 / test["B365A"]) / margin

# Baseline (training frequencies)
base = train["FTR"].value_counts(normalize=True)

print("Baseline:", grade(base["H"], base["D"], base["A"]))
print("ML model:", grade(test["ml_H"], test["ml_D"], test["ml_A"]))
print("Bookmaker:", grade(book_H, book_D, book_A))

Baseline: (np.float64(0.6567), np.float64(1.0854))
ML model: (np.float64(0.6375), np.float64(1.0554))
Bookmaker: (np.float64(0.6115), np.float64(1.0185))


In [7]:
from football import poisson_walk_forward
print("Imported!")

Imported!


In [8]:
from football import poisson_walk_forward

pairs = [("2223", "2324"), ("2324", "2425"), ("2425", "2526")]

parts = []
for old_code, new_code in pairs:
    old = data[data["Season"] == old_code]
    new = data[data["Season"] == new_code]
    parts.append(poisson_walk_forward(old, new, 2))

pois = pd.concat(parts)

data = data.merge(pois, on=["Date", "HomeTeam", "AwayTeam"], how="left")

print(data.groupby("Season")["pois_H"].count())

Season
2223      0
2324    380
2425    380
2526    380
Name: pois_H, dtype: int64


In [9]:
combo = features + ["pois_H", "pois_A"]

train2 = data[data["Season"].isin(["2324", "2425"])].dropna(subset=combo)
test2 = data[data["Season"] == "2526"].copy()
test2[features] = test2[features].fillna(train2[features].mean())

model2 = LogisticRegression(max_iter=1000)
model2.fit(train2[combo], train2["FTR"])
p = model2.predict_proba(test2[combo])

print("Train:", len(train2), "| Test:", len(test2))
print()
print("Baseline:      ", grade(base["H"], base["D"], base["A"]))
print("Poisson only:  ", grade(test2["pois_H"], test2["pois_D"], test2["pois_A"]))
print("ML form only:  ", grade(test["ml_H"], test["ml_D"], test["ml_A"]))
print("ML combined:   ", grade(p[:, 2], p[:, 1], p[:, 0]))
print("Bookmaker:     ", grade(book_H, book_D, book_A))

Train: 740 | Test: 380

Baseline:       (np.float64(0.6567), np.float64(1.0854))
Poisson only:   (np.float64(0.6367), np.float64(1.0576))
ML form only:   (np.float64(0.6375), np.float64(1.0554))
ML combined:    (np.float64(0.6264), np.float64(1.0406))
Bookmaker:      (np.float64(0.6115), np.float64(1.0185))


In [10]:
test2["comb_H"] = p[:, 2]
test2["comb_D"] = p[:, 1]
test2["comb_A"] = p[:, 0]
test2["book_H"] = book_H

def backtest(model_col):
    edge = test2[model_col] - test2["book_H"]
    bets = test2[edge > 0.05].copy()

    bets["profit"] = -1.0
    bets.loc[bets["FTR"] == "H", "profit"] = bets["B365H"] - 1

    n = len(bets)
    profit = bets["profit"].sum()
    roi = profit / n * 100
    return n, round(profit, 2), round(roi, 1)

print("Poisson only:  bets, profit, ROI% =", backtest("pois_H"))
print("ML combined:   bets, profit, ROI% =", backtest("comb_H"))

Poisson only:  bets, profit, ROI% = (109, np.float64(-13.33), np.float64(-12.2))
ML combined:   bets, profit, ROI% = (84, np.float64(-16.34), np.float64(-19.5))


In [11]:
def grade_on(results, pH, pD, pA):
    iH = (results == "H").astype(int)
    iD = (results == "D").astype(int)
    iA = (results == "A").astype(int)
    brier = ((pH - iH)**2 + (pD - iD)**2 + (pA - iA)**2).mean()
    log_loss = -np.log(pH * iH + pD * iD + pA * iA).mean()
    return round(brier, 4), round(log_loss, 4)

tr = data[data["Season"] == "2324"].dropna(subset=combo)
te = data[data["Season"] == "2425"].copy()
te[features] = te[features].fillna(tr[features].mean())

m_form = LogisticRegression(max_iter=1000).fit(tr[features], tr["FTR"])
m_comb = LogisticRegression(max_iter=1000).fit(tr[combo], tr["FTR"])
pf = m_form.predict_proba(te[features])
pc = m_comb.predict_proba(te[combo])

mg = 1 / te["B365H"] + 1 / te["B365D"] + 1 / te["B365A"]
b = tr["FTR"].value_counts(normalize=True)
res = te["FTR"]

print("Train:", len(tr), "| Test:", len(te))
print("Baseline:     ", grade_on(res, b["H"], b["D"], b["A"]))
print("Poisson only: ", grade_on(res, te["pois_H"], te["pois_D"], te["pois_A"]))
print("ML form only: ", grade_on(res, pf[:, 2], pf[:, 1], pf[:, 0]))
print("ML combined:  ", grade_on(res, pc[:, 2], pc[:, 1], pc[:, 0]))
print("Bookmaker:    ", grade_on(res, (1/te["B365H"])/mg, (1/te["B365D"])/mg, (1/te["B365A"])/mg))

Train: 365 | Test: 380
Baseline:      (np.float64(0.6581), np.float64(1.0844))
Poisson only:  (np.float64(0.5871), np.float64(0.9845))
ML form only:  (np.float64(0.625), np.float64(1.0414))
ML combined:   (np.float64(0.5927), np.float64(0.9948))
Bookmaker:     (np.float64(0.5787), np.float64(0.9708))
